In [16]:
datasets_to_process = [
    # "atc_asr_test",
    # "atc_asr_val",
    #
    "LibriSpeech-dev-clean-sint",
    # "LibriSpeech-dev-other-sint",
    #
    # "LibriSpeech-test-clean-sint",
    # "LibriSpeech-test-other-sint",
    #
    # "LibriSpeech-train-sim-100h",
    # "proc_unlab_atc_clips",
]

In [2]:
from paths import DATA_DIR

In [3]:
import os
import tempfile
import shutil
import logging
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
from scipy.signal import butter, sosfiltfilt
import soundfile as sf
from tqdm.notebook import tqdm

try:
    from pydub import AudioSegment

    PYDUB_AVAILABLE = True
except ImportError:
    PYDUB_AVAILABLE = False

log = logging.getLogger(__name__)

SOUNDFILE_FORMATS = {".wav", ".flac", ".aiff", ".aif", ".ogg"}
PYDUB_FORMATS = {".mp3", ".m4a", ".aac"}
ALL_AUDIO_FORMATS = SOUNDFILE_FORMATS | PYDUB_FORMATS


def find_audio_files(root_dir: str) -> list[Path]:
    root = Path(root_dir)
    if not root.is_dir():
        raise ValueError(f"Not a directory: {root_dir}")
    return sorted(
        p
        for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in ALL_AUDIO_FORMATS
    )


def _butter_bandpass_sos(lowcut: float, highcut: float, fs: float, order: int = 5):
    nyq = fs / 2.0
    low, high = lowcut / nyq, highcut / nyq
    if not (0 < low < high < 1):
        raise ValueError(f"Invalid cutoffs: 0 < {lowcut} < {highcut} < {fs / 2}")
    return butter(order, [low, high], btype="band", output="sos")


def apply_bandpass(
    audio: np.ndarray, sample_rate: int, lowcut: float, highcut: float, order: int = 5
) -> np.ndarray:
    sos = _butter_bandpass_sos(lowcut, highcut, sample_rate, order)
    if audio.ndim == 1:
        return sosfiltfilt(sos, audio).astype(audio.dtype)
    return np.stack(
        [sosfiltfilt(sos, audio[:, ch]) for ch in range(audio.shape[1])], axis=1
    ).astype(audio.dtype)


def _read_soundfile(path: Path):
    data, sr = sf.read(str(path), always_2d=False, dtype="float64")
    subtype = sf.info(str(path)).subtype
    return data, sr, subtype


def _write_soundfile(path: Path, data: np.ndarray, sr: int, subtype: str):
    sf.write(str(path), data, sr, subtype=subtype)


def _read_pydub(path: Path):
    seg = AudioSegment.from_file(str(path))
    sr = seg.frame_rate
    samples = np.array(seg.get_array_of_samples(), dtype=np.float64)
    samples /= float(2 ** (seg.sample_width * 8 - 1))
    if seg.channels > 1:
        samples = samples.reshape(-1, seg.channels)
    return samples, sr, seg


def _write_pydub(path: Path, data: np.ndarray, sr: int, original_seg):
    channels = 1 if data.ndim == 1 else data.shape[1]
    sw = original_seg.sample_width
    max_val = float(2 ** (sw * 8 - 1))
    flat = data.flatten() if data.ndim > 1 else data
    int_samples = (
        (flat * max_val)
        .clip(-max_val, max_val - 1)
        .astype({1: np.int8, 2: np.int16, 4: np.int32}[sw])
    )
    seg = AudioSegment(
        int_samples.tobytes(), frame_rate=sr, sample_width=sw, channels=channels
    )
    seg.export(str(path), format=path.suffix.lstrip("."))


def _process_file(
    path: Path, lowcut: float, highcut: float, order: int
) -> tuple[Path, str | None]:
    ext = path.suffix.lower()
    if ext in PYDUB_FORMATS and not PYDUB_AVAILABLE:
        return path, "pydub not installed"

    tmp_fd, tmp_path = tempfile.mkstemp(suffix=path.suffix, dir=path.parent)
    os.close(tmp_fd)
    try:
        if ext in SOUNDFILE_FORMATS:
            data, sr, subtype = _read_soundfile(path)
            filtered = apply_bandpass(data, sr, lowcut, highcut, order)
            _write_soundfile(Path(tmp_path), filtered, sr, subtype)
        else:
            data, sr, original_seg = _read_pydub(path)
            filtered = apply_bandpass(data, sr, lowcut, highcut, order)
            _write_pydub(Path(tmp_path), filtered, sr, original_seg)

        shutil.move(tmp_path, str(path))
        return path, None
    except Exception as exc:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
        return path, str(exc)


def process_directory(
    root_dir: str,
    lowcut: float = 300.0,
    highcut: float = 3400.0,
    order: int = 5,
    max_workers: int = 8,
    dry_run: bool = False,
) -> dict:
    files = find_audio_files(root_dir)
    results = {"processed": [], "errors": []}

    if dry_run:
        print(f"[dry-run] {len(files)} file(s) found in '{root_dir}'")
        return {"processed": [], "errors": [], "found": files}

    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = {
            pool.submit(_process_file, f, lowcut, highcut, order): f for f in files
        }
        with tqdm(
            total=len(files), desc=Path(root_dir).name, unit="file", leave=True
        ) as bar:
            for future in as_completed(futures):
                path, err = future.result()
                if err:
                    results["errors"].append((path, err))
                    bar.set_postfix_str(f"✗ {path.name}: {err}", refresh=True)
                else:
                    results["processed"].append(path)
                bar.update(1)

    return results


def process_datasets(
    base_dir: str,
    datasets: list[str],
    lowcut: float = 300.0,
    highcut: float = 3400.0,
    order: int = 5,
    max_workers: int = 8,
) -> dict[str, dict]:
    all_results = {}
    for dataset in tqdm(datasets, desc="Datasets", unit="dataset"):
        path = os.path.join(base_dir, dataset)
        all_results[dataset] = process_directory(
            root_dir=path,
            lowcut=lowcut,
            highcut=highcut,
            order=order,
            max_workers=max_workers,
        )
    return all_results

In [19]:
results = process_datasets(
    base_dir=str(DATA_DIR),
    datasets=datasets_to_process,
    lowcut=300,
    highcut=3400,
    max_workers=12,
)

Datasets:   0%|          | 0/1 [00:00<?, ?dataset/s]

LibriSpeech-dev-clean-sint:   0%|          | 0/3106 [00:00<?, ?file/s]

In [4]:
datasets_to_process = ["LibriSpeech-dev-clean-bpf"]

results = process_datasets(
    base_dir=str(DATA_DIR),
    datasets=datasets_to_process,
    lowcut=300,
    highcut=3400,
    max_workers=12,
)

Datasets:   0%|          | 0/1 [00:00<?, ?dataset/s]

LibriSpeech-dev-clean-bpf:   0%|          | 0/2703 [00:00<?, ?file/s]